# IMDB Sentiment Analysis

Train a binary sentiment classifier for movie reviews. Labels are `1` for positive and `0` for negative.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

DATA_DIR = Path('skills_assessment_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('sentiment_analyse/skills_assessment_data')

MODEL_PATH = Path('skills_assessment.joblib')
if not DATA_DIR.parent.samefile(Path('.')):
    MODEL_PATH = Path('sentiment_analyse/sentiment_model.joblib')

In [2]:
def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as file:
        records = json.load(file)
    return pd.DataFrame(records)

train_df = load_dataset(DATA_DIR / 'train.json')
test_df = load_dataset(DATA_DIR / 'test.json')

X_train = train_df['text'].tolist()
y_train = train_df['label'].astype(int)
X_test = test_df['text'].tolist()
y_test = test_df['label'].astype(int)

print(train_df.shape, test_df.shape)
print(y_train.value_counts().sort_index().to_dict())
print(y_test.value_counts().sort_index().to_dict())

(25000, 2) (25000, 2)
{0: 12500, 1: 12500}
{0: 12500, 1: 12500}


In [3]:
model = Pipeline([
    ('vectorizer', TfidfVectorizer(
        strip_accents='unicode',
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('classifier', LogisticRegression(C=4.0, max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test).astype(int)

In [4]:
accuracy = accuracy_score(y_test, predictions)
test_accuracy = accuracy
conf_matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
report = classification_report(y_test, predictions, labels=[0, 1], output_dict=True, zero_division=0)
metrics_df = pd.DataFrame(report).transpose()

print(f'Accuracy: {accuracy:.4f}')
print(conf_matrix)
display(metrics_df)

Accuracy: 0.8829
[[11055  1445]
 [ 1482 11018]]


,precision,recall,f1-score,support
0,0.881790,0.88440,0.883093,12500.00000
1,0.884057,0.88144,0.882746,12500.00000
accuracy,0.882920,0.88292,0.882920,0.88292
macro avg,0.882923,0.88292,0.882920,25000.00000
weighted avg,0.882923,0.88292,0.882920,25000.00000


In [5]:
joblib.dump(model, MODEL_PATH)

print(f'Model saved to {MODEL_PATH}')

Model saved to skills_assessment.joblib
